# Patent Metadata

One tidy row per US **utility** patent with identifying / descriptive fields. Primary key: `patent_id`
(shared with every other patent output in `/project/jevans/Dawoon/Science of Science/PatentView/output/`).

## Raw data (directory & structure)
PatentsView **Granted** bulk `.tsv.zip` (downloaded 2026-05-21, `data.uspto.gov/bulkdata/datasets/pvgpatdis`):
```
/project/jevans/Dawoon/Science of Science/PatentView/Granted/
├── g_patent.tsv.zip                  # patent_id, patent_type, patent_date        -> grant_year, utility filter
├── g_application.tsv.zip             # patent_id, filing_date                      -> filing_year
├── g_cpc_current.tsv.zip             # patent_id, cpc_sequence, cpc_subclass, cpc_group, cpc_type -> cpc_code(_list)
├── g_inventor_disambiguated.tsv.zip  # patent_id, inventor_sequence, inventor_id   -> inventor_list
└── g_assignee_disambiguated.tsv.zip  # patent_id, assignee_sequence, assignee_id   -> assignee_list
```

## Output columns
`patent_id, grant_year, filing_year, ref_count, cpc_code, cpc_code_list, inventor_list, assignee_list`
- `cpc_code` = primary CPC group (lowest `cpc_sequence`); `cpc_code_list` = distinct `cpc_group`, `;`-joined.
- `inventor_list`, `assignee_list` = distinct ids ordered by sequence, `;`-joined.

Output → `/project/jevans/Dawoon/Science of Science/PatentView/output/patent_metadata.parquet`.

In [1]:
import os, sys, gc, time
import numpy as np, pandas as pd
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PatentView')
import pv_common as pv
ROOT, D, OUT = pv.BASE, pv.GRANTED, pv.OUT
OUT_FP = pv.out('patent_metadata.parquet')
pv.preflight('patent_metadata')

ROOT: C:\Users\jdwoo\OneDrive\Desktop\Research\Science of Science


## 1. Grant year (utility) + filing year

In [2]:
%%time
gp = pd.read_csv(os.path.join(D, 'g_patent.tsv.zip'), sep='\t',
                 usecols=['patent_id', 'patent_type', 'patent_date'], dtype={'patent_id': str, 'patent_type': str})
gp = gp[gp['patent_type'] == 'utility'].copy()
gp['grant_year'] = pd.to_datetime(gp['patent_date'], errors='coerce').dt.year
meta = gp[['patent_id', 'grant_year']].dropna(subset=['grant_year'])
meta['grant_year'] = meta['grant_year'].astype(int)
util = set(meta['patent_id'])
ga = pd.read_csv(os.path.join(D, 'g_application.tsv.zip'), sep='\t',
                 usecols=['patent_id', 'filing_date'], dtype={'patent_id': str})
ga = ga[ga['patent_id'].isin(util)]
ga['filing_year'] = pd.to_datetime(ga['filing_date'], errors='coerce').dt.year
fy = ga.dropna(subset=['filing_year']).groupby('patent_id')['filing_year'].min().astype(int)
meta = meta.merge(fy.rename('filing_year'), on='patent_id', how='left')
print(f'utility patents: {len(meta):,}'); del gp, ga; gc.collect()

<timed exec>:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.


utility patents: 8,531,961


CPU times: total: 31.9 s
Wall time: 32.6 s


46

## 2. ref_count (number of US-patent backward references)

In [3]:
%%time
import time
parts, t0 = [], time.time()
for ch in pd.read_csv(os.path.join(D, 'g_us_patent_citation.tsv.zip'), sep='\t',
                      usecols=['patent_id', 'citation_patent_id'], dtype=str, chunksize=5_000_000):
    ch = ch.dropna(subset=['citation_patent_id'])
    ch = ch[ch['patent_id'].isin(util)]
    if len(ch): parts.append(ch['patent_id'].value_counts())
ref_count = pd.concat(parts).groupby(level=0).sum()
meta = meta.merge(ref_count.rename('ref_count'), left_on='patent_id', right_index=True, how='left')
meta['ref_count'] = meta['ref_count'].fillna(0).astype(int)
print(f'[{time.time()-t0:.0f}s] ref_count: mean {meta.ref_count.mean():.1f}')

[176s] ref_count: mean 15.8
CPU times: total: 2min 52s
Wall time: 2min 55s


## 3. CPC (primary + list), inventors, assignees

In [4]:
%%time
cpc = pd.read_csv(os.path.join(D, 'g_cpc_current.tsv.zip'), sep='\t',
                  usecols=['patent_id', 'cpc_sequence', 'cpc_group'], dtype={'patent_id': str, 'cpc_group': str})
cpc = cpc[cpc['patent_id'].isin(util)].dropna(subset=['cpc_group'])
cpc['cpc_sequence'] = pd.to_numeric(cpc['cpc_sequence'], errors='coerce')
primary = cpc.sort_values(['patent_id', 'cpc_sequence']).drop_duplicates('patent_id', keep='first') \
             .set_index('patent_id')['cpc_group'].rename('cpc_code')
cpc_list = cpc.sort_values(['patent_id', 'cpc_sequence']).drop_duplicates(['patent_id', 'cpc_group']) \
              .groupby('patent_id')['cpc_group'].agg(';'.join).rename('cpc_code_list')

def joined_list(fn, seq_col, id_col):
    d = pd.read_csv(os.path.join(D, fn), sep='\t', usecols=['patent_id', seq_col, id_col],
                    dtype={'patent_id': str, id_col: str})
    d = d[d['patent_id'].isin(util)].dropna(subset=[id_col])
    d[seq_col] = pd.to_numeric(d[seq_col], errors='coerce')
    return (d.sort_values(['patent_id', seq_col]).drop_duplicates(['patent_id', id_col])
             .groupby('patent_id')[id_col].agg(';'.join))

# NOTE: the disambiguated files use *_id / *_sequence column names (inventor_id, assignee_id)
inv = joined_list('g_inventor_disambiguated.tsv.zip', 'inventor_sequence', 'inventor_id').rename('inventor_list')
asg = joined_list('g_assignee_disambiguated.tsv.zip', 'assignee_sequence', 'assignee_id').rename('assignee_list')
# primary/cpc_list/inv/asg are Series indexed by patent_id -> merge on index (proven pattern)
for s in (primary, cpc_list, inv, asg):
    meta = meta.merge(s, left_on='patent_id', right_index=True, how='left')
del cpc; gc.collect()
print('columns:', list(meta.columns))

columns: ['patent_id', 'grant_year', 'filing_year', 'ref_count', 'cpc_code', 'cpc_code_list', 'inventor_list', 'assignee_list']
CPU times: total: 8min 20s
Wall time: 8min 30s


## 4. Save + example

In [5]:
meta = meta[['patent_id', 'grant_year', 'filing_year', 'ref_count', 'cpc_code', 'cpc_code_list', 'inventor_list', 'assignee_list']]
meta.to_parquet(OUT_FP, index=False)
print(f'WROTE {OUT_FP}  ({len(meta):,} rows, {len(meta.columns)} cols)')
for c in meta.columns:
    print(f'  {c:16s} non-null {meta[c].notna().mean()*100:5.1f}%')
display(meta.head(10))

WROTE C:\Users\jdwoo\OneDrive\Desktop\Research\Science of Science\notebook\patent\output\patent_metadata.parquet  (8,531,961 rows, 8 cols)
  patent_id        non-null 100.0%
  grant_year       non-null 100.0%
  filing_year      non-null 100.0%
  ref_count        non-null 100.0%


  cpc_code         non-null  99.8%


  cpc_code_list    non-null  99.8%


  inventor_list    non-null 100.0%


  assignee_list    non-null  90.8%


,patent_id,grant_year,filing_year,ref_count,cpc_code,cpc_code_list,inventor_list,assignee_list
0,10000000,2018,2015.0,2,G01S7/4863,G01S7/4863;G01S7/4865;G01S7/4914;G01S7/4917;G0...,fl:jo_ln:marron-5,a45783ae-9cec-49bc-bc2e-7b064aaa5907
1,10000001,2018,2015.0,6,B29C45/64,B29C45/64;B29C45/76;G05B19/182;G05B19/402;B29C...,fl:su_ln:lee-252;fl:hy_ln:yu-60,c7fdbc77-28f0-4a1c-9f08-c13171f1009c
2,10000002,2018,2014.0,3,B29C48/21,B29C48/21;B29C48/022;B29C48/08;B29C48/71;B29D3...,fl:yu_ln:kim-61;fl:si_ln:kim-38;fl:do_ln:choi-...,5b722cc6-9d8f-46bd-83cb-7838a17e5201
3,10000003,2018,2013.0,4,B29C49/04102,B29C49/04102;B29C49/20;B29C49/22;B29C49/4802;B...,fl:gu_ln:bergmann-3;fl:ca_ln:elsasser-1;fl:cr_...,b153659d-bd6c-4cfd-8241-d22de6410750
4,10000004,2018,2015.0,2,B29C51/02,B29C51/02;B29C48/0017;B29C48/05;B29C48/10;B29C...,fl:mi_ln:zubiriaelizondo-1;fl:jo_ln:valadezlop...,7b1383b0-7643-4186-9c9a-c4d9f9e8156f
5,10000005,2018,2012.0,6,B29C51/04,B29C51/04;B29C51/262;B29C51/082;B29C51/10;B29C...,fl:ke_ln:katou-9;fl:ka_ln:oda-24,32e04930-7fff-4c0c-a861-989689853f13
6,10000006,2018,2015.0,0,B29C51/36,B29C51/36;B29C51/082;B29C51/087;B29C51/10;B60R...,fl:ma_ln:saelen-1,bbef70eb-0952-4b89-997a-54831dcadcd5
7,10000007,2018,2016.0,72,B29C57/04,B29C57/04;B29K2023/0691;B29L2023/22,fl:co_ln:dickert-1;fl:al_ln:huber-4;fl:ja_ln:k...,5f83d790-2a6a-421e-8d6c-6da6e6869187
8,10000008,2018,2014.0,17,B29C61/025,B29C61/025;B29C33/405;B29C33/42;B29C61/02;A44C...,fl:li_ln:caspi-2,65aa142b-2a7a-4079-aae5-f99079ca9fa0
9,10000009,2018,2015.0,5,B29C64/112,B29C64/112;B29C64/106;B29C64/118;B29C64/255;B2...,fl:na_ln:maier-2,NaN
